# N-CMAPSS DS05 — representation, then BO and PSO

## Why representation comes first

The v2 ablation found that **every** component made things worse: bidirectional LSTM +0.453,
attention +0.545, multi-scale +0.416, max-pooling +0.372, dropout +1.624. That is not what one
expects from four independent mechanisms, and the explanation is in the input shape.

Our input is `(18, 1)` — eighteen **features** reshaped as a sequence. There is no time axis; each
sample is a single flight-second snapshot. So in our models a bidirectional LSTM reads feature 1
to 18 forwards and backwards, attention weights feature *positions*, and multi-scale convolution
mixes adjacent feature *columns*. Every component that failed is a temporal mechanism with no time
to act on.

The benchmarks we are behind are all sequence models — GRU, LSTM, BiLSTM, Transformer — and MSTSDN
is literally *Multi-Scale Temporal-Spatial*. Those names only mean anything on a real time axis, so
they must window over flight history. The data supports it easily: unit 1 has ~9,200 rows per
flight, roughly 915 after 1-in-10 subsampling.

**Part 1** compares the snapshot representation against genuine temporal windows.
**Part 2** runs Bayesian optimization and particle swarm on whichever wins, anchored so the
baseline is in the population and cannot be lost, with test logged per trial for diagnosis only.

A caution carried from FD003: across the six v2 ablation configurations, Spearman(validation, test)
was **+0.486, p = 0.33**, and validation units 5-6 are systematically easier than test units 7-10
by 1.6-1.9 RMSE. A weakly-aligned objective is what inverted every FD003 search, so Part 2 logs the
correlation as it goes.

In [ ]:
import os, json, gc, time, random, math
os.environ['PYTHONHASHSEED']='42'
import numpy as np, pandas as pd, h5py
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel, WhiteKernel
from scipy.stats import norm, spearmanr
import tensorflow as tf
for _g in tf.config.list_physical_devices('GPU'):
    try: tf.config.experimental.set_memory_growth(_g,True)
    except RuntimeError as _e: print('memory growth:',_e)
from tensorflow.keras import backend as K, regularizers
from tensorflow.keras.layers import (Input, Conv1D, MaxPooling1D, Dropout, Dense, LSTM,
                                     Bidirectional, Concatenate, Flatten, Softmax, Multiply,
                                     Lambda, GlobalAveragePooling1D, GlobalMaxPooling1D)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
def set_seeds(s):
    os.environ['PYTHONHASHSEED']=str(s); random.seed(s); np.random.seed(s)
    tf.keras.utils.set_random_seed(s)
def preflight():
    if not tf.config.list_physical_devices('GPU'): print('CPU mode'); return False
    try:
        x=np.random.rand(8,10,3).astype('float32'); y=np.random.rand(8,1).astype('float32')
        i=Input(shape=(10,3)); o=Dense(1)(GlobalAveragePooling1D()(Conv1D(4,3,padding='same')(i)))
        m=Model(i,o); m.compile('adam','mse'); m.fit(x,y,epochs=1,verbose=0); K.clear_session()
        print('GPU preflight passed'); return True
    except Exception as e:
        print('GPU preflight FAILED (%s)'%type(e).__name__); K.clear_session(); return False
USE_GPU=preflight(); DEVICE='/GPU:0' if USE_GPU else '/CPU:0'
try: tf.config.experimental.enable_op_determinism()
except Exception as e: print('determinism:',e)
print('TF',tf.__version__,'| device',DEVICE)

In [ ]:
SEED=42; SUBSAMPLE=10; VAL_UNITS=[5,6]; N_SEEDS=3
set_seeds(SEED)

def read_sub(dset, step, dtype=np.float32, chunk=400_000):
    """Read an HDF5 dataset in sequential chunks, subsampling and casting inside the loop.
       Never materialises the full float64 array - that is what exhausted RAM in v3.0."""
    n=dset.shape[0]; out=[]
    for a in range(0,n,chunk):
        b=min(a+chunk,n)
        blk=dset[a:b]
        first=(-a)%step                      # keep the global 1-in-step alignment
        if first < len(blk): out.append(blk[first::step].astype(dtype,copy=False))
    return np.concatenate(out) if out else np.empty((0,dset.shape[1]),dtype)

S=SUBSAMPLE
with h5py.File('N-CMAPSS_DS05.h5','r') as h:
    Wd=read_sub(h['W_dev'],S);   Xd=read_sub(h['X_s_dev'],S)
    Ad=read_sub(h['A_dev'],S);   Yd=read_sub(h['Y_dev'],S)
    Wt=read_sub(h['W_test'],S);  Xt=read_sub(h['X_s_test'],S)
    At=read_sub(h['A_test'],S);  Yt=read_sub(h['Y_test'],S)
    FEATS=[str(np.array(x).astype('U20')) for x in np.array(h['W_var'])]+\
          [str(np.array(x).astype('U20')) for x in np.array(h['X_s_var'])]
NF=len(FEATS)
mb=lambda *a: sum(x.nbytes for x in a)/1e6
print('loaded subsampled 1-in-%d as float32: %.0f MB total'%(S,mb(Wd,Xd,Ad,Yd,Wt,Xt,At,Yt)))

Fd=np.hstack([Wd,Xd]).astype(np.float32); ud=Ad[:,0].astype(int); cd=Ad[:,1].astype(int)
yd=Yd.ravel().astype(np.float32)
Ft=np.hstack([Wt,Xt]).astype(np.float32); ut=At[:,0].astype(int); ct=At[:,1].astype(int)
yt=Yt.ravel().astype(np.float32)
del Wd,Xd,Ad,Yd,Wt,Xt,At,Yt; gc.collect()

tr_m=~np.isin(ud,VAL_UNITS); va_m=np.isin(ud,VAL_UNITS)
SC=MinMaxScaler().fit(Fd[tr_m])
Fd=SC.transform(Fd).astype(np.float32); Ft=SC.transform(Ft).astype(np.float32)
TRIVIAL=float(np.sqrt(mean_squared_error(yt,np.full_like(yt,yd[tr_m].mean()))))
assert not (set(ud[tr_m])&set(VAL_UNITS)) and not (set(ud)&set(ut))
print('train units %s | val %s | test %s | trivial floor %.3f'
      %(sorted(set(ud[tr_m])),VAL_UNITS,sorted(set(ut)),TRIVIAL))

def snapshot(F,u,c,y,mask):
    return F[mask].reshape(-1,NF,1), y[mask]

def windows(F,u,c,y,mask,W,stride):
    """Windows of W consecutive rows WITHIN one flight (unit, cycle). RUL is constant per flight."""
    X,Y=[],[]
    uu=u[mask]; cc=c[mask]; FF=F[mask]; yy=y[mask]
    order=np.lexsort((cc,uu)); uu,cc,FF,yy=uu[order],cc[order],FF[order],yy[order]
    key=uu.astype(np.int64)*100000+cc
    bounds=np.flatnonzero(np.diff(key))+1
    for a,b in zip(np.r_[0,bounds],np.r_[bounds,len(key)]):
        A=FF[a:b]; L=yy[a:b]
        for e in range(W,len(A)+1,stride):
            X.append(A[e-W:e]); Y.append(L[e-1])
    return np.asarray(X,np.float32), np.asarray(Y,np.float32)

REPS={'snapshot (18,1)':dict(kind='snap'),
      'window 25 (stride 12)':dict(kind='win',W=25,stride=12),
      'window 50 (stride 25)':dict(kind='win',W=50,stride=25),
      'window 100 (stride 50)':dict(kind='win',W=100,stride=50)}
DATA={}
for nm,cfg in REPS.items():
    if cfg['kind']=='snap':
        a=snapshot(Fd,ud,cd,yd,tr_m); b=snapshot(Fd,ud,cd,yd,va_m); t=snapshot(Ft,ut,ct,yt,np.ones(len(yt),bool))
        shape=(NF,1)
    else:
        W,S=cfg['W'],cfg['stride']
        a=windows(Fd,ud,cd,yd,tr_m,W,S); b=windows(Fd,ud,cd,yd,va_m,W,S)
        t=windows(Ft,ut,ct,yt,np.ones(len(yt),bool),W,S); shape=(W,NF)
    DATA[nm]=dict(Xtr=a[0],ytr=a[1],Xva=b[0],yva=b[1],Xte=t[0],yte=t[1],shape=shape)
    print('%-24s input %-10s train %8s | val %7s | test %7s'
          %(nm,str(shape),format(len(a[0]),','),format(len(b[0]),','),format(len(t[0]),',')))

## Model

The same parallel CNN-LSTM, but now the convolution runs over **time** and the LSTM reads a real
sequence whenever the input is windowed.

In [ ]:
def attn_pool(seq,s):
    w=Softmax(axis=1)(Dense(1,activation='tanh',
        kernel_initializer=tf.keras.initializers.GlorotUniform(seed=s))(seq))
    return Lambda(lambda x: K.sum(x,axis=1))(Multiply()([seq,w]))

def build(shape,seed=SEED,filters=(96,128,128,32),lstm_units=80,dense=(192,96),pool='avg',
          dropout=0.0,l2reg=0.0,lr=1e-3,bidirectional=False,attention=False,multiscale=False):
    kw=dict(kernel_initializer=tf.keras.initializers.GlorotUniform(seed=seed),
            kernel_regularizer=regularizers.l2(l2reg) if l2reg>0 else None)
    inp=Input(shape=shape); c=inp
    for i,f in enumerate(filters):
        if multiscale:
            c=Concatenate()([Conv1D(f,k,padding='same',activation='relu',**kw)(c) for k in (3,5,7)])
        else:
            c=Conv1D(f,3,padding='same',activation='relu',**kw)(c)
        if dropout>0: c=Dropout(dropout,seed=seed)(c)
        if i<2 and int(c.shape[1])>2: c=MaxPooling1D(2,padding='same')(c)
    cv=(GlobalMaxPooling1D() if pool=='max' else GlobalAveragePooling1D())(c)
    cell=LSTM(int(lstm_units),return_sequences=attention,**kw)
    r=Bidirectional(cell)(inp) if bidirectional else cell(inp)
    if attention: r=attn_pool(r,seed)
    h=Concatenate()([cv,r])
    for d in dense:
        h=Dense(int(d),activation='relu',**kw)(h)
        if dropout>0: h=Dropout(dropout,seed=seed)(h)
    m=Model(inp,Dense(1,activation='linear',**kw)(h))
    m.compile(optimizer=tf.keras.optimizers.Adam(lr),loss='mse'); return m

def run(rep,seed,epochs=300,patience=30,batch=256,**kw):
    d=DATA[rep]; K.clear_session(); gc.collect(); set_seeds(seed); t0=time.time()
    with tf.device(DEVICE):
        m=build(d['shape'],seed=seed,**kw)
        h=m.fit(d['Xtr'],d['ytr'],validation_data=(d['Xva'],d['yva']),epochs=epochs,
                batch_size=int(batch),verbose=0,
                callbacks=[EarlyStopping('val_loss',patience=patience,restore_best_weights=True),
                           ReduceLROnPlateau('val_loss',factor=0.5,patience=patience//2,min_lr=1e-6)])
        rm=lambda X,y: float(np.sqrt(mean_squared_error(y,m.predict(X,batch_size=4096,verbose=0))))
        v=rm(d['Xva'],d['yva']); t=rm(d['Xte'],d['yte'])
    return dict(val=v,test=t,params=int(m.count_params()),epochs=len(h.history['loss']),
                secs=round(time.time()-t0))

## Part 1 — does a real time axis help?

In [ ]:
R1=[]; t0=time.time()
for nm in REPS:
    ep = 190 if 'snapshot' in nm else 300
    rs=[run(nm,SEED+s,epochs=ep,patience=ep//8) for s in range(N_SEEDS)]
    te=np.array([r['test'] for r in rs]); va=np.array([r['val'] for r in rs])
    R1.append(dict(rep=nm,test_mean=te.mean(),test_sd=te.std(ddof=1),val_mean=va.mean(),
                   params=rs[0]['params'],epochs=np.mean([r['epochs'] for r in rs]),
                   secs=np.mean([r['secs'] for r in rs]),test_all=[round(x,3) for x in te]))
    print('%-24s test %6.3f +/- %5.3f | val %6.3f | ep %5.1f/%d | %4.0fs/run'
          %(nm,te.mean(),te.std(ddof=1),va.mean(),np.mean([r['epochs'] for r in rs]),ep,
            np.mean([r['secs'] for r in rs])))
P1=pd.DataFrame(R1); pd.set_option('display.width',200)
print('\n'+P1[['rep','test_mean','test_sd','val_mean','params','epochs','secs']].to_string(index=False))
BEST_REP=P1.sort_values('val_mean').iloc[0]['rep']
snap=P1[P1.rep.str.contains('snapshot')].test_mean.iloc[0]
print('\ntrivial %.3f | snapshot %.3f | best representation %s at %.3f (%+.3f)'
      %(TRIVIAL,snap,BEST_REP,P1.test_mean.min(),P1.test_mean.min()-snap))
print('part 1 finished in %.0f s'%(time.time()-t0))

## Part 1b — re-run the ablation on the winning representation

The v2 ablation found every component harmful. If that was caused by the missing time axis, the
same components should behave differently now.

In [ ]:
LAD=[('R0','baseline',dict()),('R1','+ bidirectional',dict(bidirectional=True)),
     ('R2','+ attention',dict(bidirectional=True,attention=True)),
     ('R3','+ multi-scale',dict(bidirectional=True,attention=True,multiscale=True)),
     ('R4','max pooling',dict(pool='max'))]
ep = 190 if 'snapshot' in BEST_REP else 300
AB=[]
for rid,label,kw in LAD:
    rs=[run(BEST_REP,SEED+s,epochs=ep,patience=ep//8,**kw) for s in range(N_SEEDS)]
    te=np.array([r['test'] for r in rs]); va=np.array([r['val'] for r in rs])
    AB.append(dict(id=rid,config=label,test_mean=te.mean(),test_sd=te.std(ddof=1),
                   val_mean=va.mean(),params=rs[0]['params']))
    print('%-3s %-18s test %6.3f +/- %5.3f | val %6.3f | params %9s'
          %(rid,label,te.mean(),te.std(ddof=1),va.mean(),format(rs[0]['params'],',')))
A2=pd.DataFrame(AB)
A2['delta']=(A2.test_mean-A2.test_mean.iloc[0]).round(3)
A2['noise']=np.sqrt(A2.test_sd**2+A2.test_sd.iloc[0]**2).round(3)
A2['verdict']=np.where(A2.delta.abs()>A2.noise,np.where(A2.delta<0,'BETTER','worse'),'within noise')
A2.loc[0,'verdict']='-'
print('\n'+A2[['id','config','test_mean','test_sd','delta','noise','verdict']].to_string(index=False))
print('\nv2 on the snapshot representation: every component was worse (+0.37 to +1.62)')
r,p=spearmanr(A2.val_mean,A2.test_mean)
print('Spearman(val,test) on this representation: %+.3f (p=%.3f)   [snapshot was +0.486, p=0.33]'%(r,p))

## Part 2 — Bayesian optimization and particle swarm

Both are written here so the initial design is controlled. The baseline is evaluated and enters the
population of each; neither can return a configuration worse than its starting point without that
being visible. Test is logged per trial and never enters selection.

In [ ]:
SPACE=[('f1',32,192,'int'),('f2',32,192,'int'),('f3',16,160,'int'),('f4',16,96,'int'),
       ('lstm_units',16,192,'int'),('d1',64,320,'int'),('d2',32,192,'int'),
       ('dropout',0.0,0.4,'float'),('lr',1e-4,1e-2,'log'),('l2reg',1e-8,1e-2,'log')]
DEFAULTS=dict(f1=96,f2=128,f3=128,f4=32,lstm_units=80,d1=192,d2=96,dropout=0.0,lr=1e-3,l2reg=1e-8)
BO_INIT,BO_ITERS = 6,14
PSO_PART,PSO_ITERS = 6,3
SEARCH_SEEDS, CONFIRM_SEEDS, SHORTLIST = 2,5,5
EP = 190 if 'snapshot' in BEST_REP else 300
D=len(SPACE)
def kw_of(v):
    kw={}
    for (n,lo,hi,k),x in zip(SPACE,np.clip(v,0,1)):
        kw[n]=int(round(lo+x*(hi-lo))) if k=='int' else (
              float(math.exp(math.log(lo)+x*(math.log(hi)-math.log(lo)))) if k=='log'
              else float(lo+x*(hi-lo)))
    return kw
def enc(kw):
    v=[]
    for n,lo,hi,k in SPACE:
        x=kw[n]
        v.append((math.log(max(x,lo))-math.log(lo))/(math.log(hi)-math.log(lo)) if k=='log' else (x-lo)/(hi-lo))
    return np.clip(np.array(v,float),0,1)
def to_model(kw):
    return dict(filters=(kw['f1'],kw['f2'],kw['f3'],kw['f4']),lstm_units=kw['lstm_units'],
                dense=(kw['d1'],kw['d2']),dropout=kw['dropout'],lr=kw['lr'],l2reg=kw['l2reg'])
BASE_VEC=enc(DEFAULTS); dist=lambda v: float(np.linalg.norm(np.asarray(v)-BASE_VEC))
LOG=[]; CACHE={}
def fitness(v,tag=''):
    key=tuple(np.round(v,5))
    if key in CACHE: return CACHE[key]
    kw=kw_of(v); rs=[run(BEST_REP,SEED+s,epochs=EP,patience=EP//8,**to_model(kw)) for s in range(SEARCH_SEEDS)]
    val=float(np.mean([r['val'] for r in rs])); te=float(np.mean([r['test'] for r in rs]))
    CACHE[key]=val; LOG.append(dict(tag=tag,val=val,test=te,dist=dist(v),
                                    params=rs[0]['params'],**kw))
    print('   %-8s val %6.3f | test %6.3f | dist %.2f'%(tag,val,te,dist(v)))
    return val
print('cost estimate: %d BO + %d PSO evals x %d seeds at ~%.0f s/run'
      %(BO_INIT+BO_ITERS,PSO_PART*(1+PSO_ITERS),SEARCH_SEEDS,P1[P1.rep==BEST_REP].secs.iloc[0]))
BASE_VAL=fitness(BASE_VEC,'baseline')

In [ ]:
def gp_bo(n_init,n_iter,seed):
    r=np.random.RandomState(seed)
    X=[BASE_VEC.copy()]+list(r.rand(n_init,D)); y=[CACHE[tuple(np.round(BASE_VEC,5))]]
    y+= [fitness(x,'bo-init') for x in X[1:]]
    for it in range(n_iter):
        gp=GaussianProcessRegressor(
            kernel=ConstantKernel(1.0)*Matern(length_scale=np.ones(D),nu=2.5)+WhiteKernel(1e-2),
            normalize_y=True,n_restarts_optimizer=2,random_state=seed)
        gp.fit(np.array(X),np.array(y))
        cand=np.random.RandomState(seed+it).rand(4000,D)
        mu,sd=gp.predict(cand,return_std=True); best=min(y)
        z=(best-mu)/np.maximum(sd,1e-9); ei=(best-mu)*norm.cdf(z)+sd*norm.pdf(z)
        x=cand[int(np.argmax(ei))]; X.append(x); y.append(fitness(x,'bo'))
        print('  BO %2d: best %.3f'%(it+1,min(y)))
    return np.array(X),np.array(y)
print('=== Bayesian optimization ===')
t0=time.time(); BX,BY=gp_bo(BO_INIT,BO_ITERS,SEED); print('BO done in %.0f s'%(time.time()-t0))

def pso(n_part,iters,seed,w=0.72,c1=1.5,c2=1.5):
    r=np.random.RandomState(seed)
    X=r.rand(n_part,D); X[0]=BASE_VEC.copy()          # baseline is particle 0
    V=r.uniform(-0.1,0.1,(n_part,D))
    f=np.array([fitness(x,'pso-init') for x in X]); P=X.copy(); Pf=f.copy()
    G=P[int(np.argmin(Pf))].copy(); Gf=float(Pf.min())
    print('  PSO init best %.3f (baseline particle %.3f)'%(Gf,Pf[0]))
    for it in range(iters):
        r1,r2=r.rand(n_part,D),r.rand(n_part,D)
        V=np.clip(w*V+c1*r1*(P-X)+c2*r2*(G-X),-0.25,0.25); X=np.clip(X+V,0,1)
        f=np.array([fitness(x,'pso') for x in X])
        imp=f<Pf; P[imp]=X[imp]; Pf[imp]=f[imp]
        if Pf.min()<Gf: Gf=float(Pf.min()); G=P[int(np.argmin(Pf))].copy()
        print('  PSO %d: best %.3f'%(it+1,Gf))
    return G,Gf
print('\n=== particle swarm ===')
t0=time.time(); PG,PGf=pso(PSO_PART,PSO_ITERS,SEED); print('PSO done in %.0f s'%(time.time()-t0))
L=pd.DataFrame(LOG); L.to_csv('ncmapss_search_log.csv',index=False)
r,p=spearmanr(L.val,L.test)
print('\nSpearman(objective, test) across %d evaluated configurations: %+.3f (p=%.4f)'%(len(L),r,p))

## Two-stage decision

In [ ]:
short=L.nsmallest(SHORTLIST,'val')
cands=[('baseline',DEFAULTS)]+[('cfg %d'%i,{n:row[n] for n,_,_,_ in SPACE})
                               for i,row in enumerate(short.to_dict('records'))]
S2=[]
print('re-evaluating %d configurations on %d fresh seeds each'%(len(cands),CONFIRM_SEEDS))
for nm,kw in cands:
    rs=[run(BEST_REP,SEED+900+s,epochs=EP,patience=EP//8,**to_model(kw)) for s in range(CONFIRM_SEEDS)]
    va=np.array([r['val'] for r in rs]); te=np.array([r['test'] for r in rs])
    S2.append(dict(name=nm,val=va.mean(),val_sd=va.std(ddof=1),test=te.mean(),
                   test_sd=te.std(ddof=1),params=rs[0]['params']))
    print('  %-10s val %6.3f +/- %5.3f | test %6.3f +/- %5.3f'
          %(nm,va.mean(),va.std(ddof=1),te.mean(),te.std(ddof=1)))
T2=pd.DataFrame(S2)
base=T2[T2.name=='baseline'].iloc[0]; win=T2.nsmallest(1,'val').iloc[0]
d=win['test']-base['test']; nz=math.sqrt(win['test_sd']**2+base['test_sd']**2)
print('\nselected on validation: %s | test %.3f vs baseline %.3f -> %+.3f against noise %.3f'
      %(win['name'],win['test'],base['test'],d,nz))
print('DECISION: %s'%('adopt %s'%win['name'] if d<-nz else 'tie - keep the baseline'))
T2.to_csv('ncmapss_twostage.csv',index=False)
json.dump(dict(rep=BEST_REP,part1=R1,ablation=AB,search=LOG,twostage=T2.to_dict('records'),
               trivial=TRIVIAL,spearman=[float(r),float(p)]),
          open('ncmapss_search_results.json','w'),indent=2,default=str)
print('saved ncmapss_search_log.csv, ncmapss_twostage.csv, ncmapss_search_results.json')